
## DATA LOADING ##

In [18]:
import pandas as pd

df=pd.read_csv('/Users/mohitverma/Downloads/flights.csv')


In [12]:
df.head()

,index,airline,date_of_journey,Source,destination,route,dep_time,Arrival_time,Duration,Total_stops,Additional_info,Price
0,1,AirAsia India,2024-02-25,Guwahati,Indore,GUW-NAG-MUM-RAI-IND,01:39,14:15,3h 15min,3 stops,Red-eye flight,14170
1,2,TruJet,2021-03-01,Nagpur,Pune,NAG-PUN,06:48,03:21,1h 0min,non-stop,1 Long layover,10295
2,3,Star Air,2024-04-11,Ahmedabad,Varanasi,AHM-PUN-BAN-VAR,08:54,15:50,14h 30min,2 stops,In-flight meal not included,9165
3,4,SpiceJet,2019-03-13,Patna,Indore,PAT-LUC-GOA-BHU-IND,18:09,08:23,7h 15min,3 stops,No info,9944
4,5,SpiceJet,2023-06-22,Raipur,Mumbai,RAI-GOA-MUM,15:05,05:52,15h 30min,1 stop,1 Long layover,9474


In [14]:
print("-"*30)

------------------------------


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   index            15000 non-null  int64 
 1   airline          15000 non-null  object
 2   date_of_journey  15000 non-null  object
 3   Source           15000 non-null  object
 4   destination      15000 non-null  object
 5   route            15000 non-null  object
 6   dep_time         15000 non-null  object
 7   Arrival_time     15000 non-null  object
 8   Duration         15000 non-null  object
 9   Total_stops      15000 non-null  object
 10  Additional_info  15000 non-null  object
 11  Price            15000 non-null  int64 
dtypes: int64(2), object(10)
memory usage: 1.4+ MB


## DATA CLEANING ##


In [23]:
#1. 'date_of_journey to datetime'

df['date_of_journey'] = pd.to_datetime(df['date_of_journey'])

In [27]:
#2 Extracting Days and Months
df['day']=df['date_of_journey'].dt.day
df['month']= df['date_of_journey'].dt.month

In [29]:
#3 Clean 'Total_stops', replacing 'non_stop'with 0 and extract numbers from strings like '2 stops'
df['Total_stops']=df['Total_stops'].replace('non-stop','0')
df['Total_stops']=df['Total_stops'].str.extract('(\d+)').fillna(0).astype(int)

<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
/var/folders/00/w2w6bgwn7fz26tqx6lhkl5480000gn/T/ipykernel_92732/1451927508.py:3: SyntaxWarning: invalid escape sequence '\d'
  df['Total_stops']=df['Total_stops'].str.extract('(\d+)').fillna(0).astype(int)


In [31]:
# 4. Convert 'Duration' to total minutes
def duration_to_minutes(duration):
    hours = 0
    mins = 0
    parts = duration.split()
    for p in parts:
        if 'h' in p:
            hours = int(p.replace('h', ''))
        elif 'min' in p:
            mins = int(p.replace('min', ''))
    return (hours * 60) + mins

df['duration_total_mins'] = df['Duration'].apply(duration_to_minutes)

In [33]:
df.head(5)

,index,airline,date_of_journey,Source,destination,route,dep_time,Arrival_time,Duration,Total_stops,Additional_info,Price,day,month,duration_total_mins
0,1,AirAsia India,2024-02-25,Guwahati,Indore,GUW-NAG-MUM-RAI-IND,01:39,14:15,3h 15min,3,Red-eye flight,14170,25,2,195
1,2,TruJet,2021-03-01,Nagpur,Pune,NAG-PUN,06:48,03:21,1h 0min,0,1 Long layover,10295,1,3,60
2,3,Star Air,2024-04-11,Ahmedabad,Varanasi,AHM-PUN-BAN-VAR,08:54,15:50,14h 30min,2,In-flight meal not included,9165,11,4,870
3,4,SpiceJet,2019-03-13,Patna,Indore,PAT-LUC-GOA-BHU-IND,18:09,08:23,7h 15min,3,No info,9944,13,3,435
4,5,SpiceJet,2023-06-22,Raipur,Mumbai,RAI-GOA-MUM,15:05,05:52,15h 30min,1,1 Long layover,9474,22,6,930


In [35]:
# 1. Standardize 'Additional_info'
# Convert to lowercase and strip whitespace to ensure 'No info' is consistent
df['Additional_info'] = df['Additional_info'].str.lower().str.strip()
df['Additional_info'] = df['Additional_info'].replace('no info', 'none')

# 2. Drop the redundant columns we replaced
# We keep 'date_of_journey' for Power BI, but 'Duration' is now redundant
cols_to_drop = ['Duration', 'index'] # 'index' is usually redundant in SQL
df_final = df.drop(columns=cols_to_drop)

# 3. Export the final cleaned dataset
df_final.to_csv('cleaned_flights.csv', index=False)

print("✅ Step 3 Complete!")
print(f"Final shape of data: {df_final.shape}")
print("\nFinal column list for SQL:")
print(df_final.columns.tolist())

# Preview the cleaned text
print("\nUnique values in Additional_info:")
print(df_final['Additional_info'].unique()[:5])

✅ Step 3 Complete!
Final shape of data: (15000, 13)

Final column list for SQL:
['airline', 'date_of_journey', 'Source', 'destination', 'route', 'dep_time', 'Arrival_time', 'Total_stops', 'Additional_info', 'Price', 'day', 'month', 'duration_total_mins']

Unique values in Additional_info:
['red-eye flight' '1 long layover' 'in-flight meal not included' 'none'
 'limited seats']


In [37]:
df.head(5)

,index,airline,date_of_journey,Source,destination,route,dep_time,Arrival_time,Duration,Total_stops,Additional_info,Price,day,month,duration_total_mins
0,1,AirAsia India,2024-02-25,Guwahati,Indore,GUW-NAG-MUM-RAI-IND,01:39,14:15,3h 15min,3,red-eye flight,14170,25,2,195
1,2,TruJet,2021-03-01,Nagpur,Pune,NAG-PUN,06:48,03:21,1h 0min,0,1 long layover,10295,1,3,60
2,3,Star Air,2024-04-11,Ahmedabad,Varanasi,AHM-PUN-BAN-VAR,08:54,15:50,14h 30min,2,in-flight meal not included,9165,11,4,870
3,4,SpiceJet,2019-03-13,Patna,Indore,PAT-LUC-GOA-BHU-IND,18:09,08:23,7h 15min,3,none,9944,13,3,435
4,5,SpiceJet,2023-06-22,Raipur,Mumbai,RAI-GOA-MUM,15:05,05:52,15h 30min,1,1 long layover,9474,22,6,930
